# 🌿 BioEcoInt — Pipeline de Treinamento Local & SOTA Benchmark (Workstation GPU)
> **Laboratório IFMaker / BioEcoInt** — *Engenharia de Software & Machine Learning Aplicada à Fauna Amazônica*
> 
> Este notebook executa localmente nesta estação de trabalho a **Bateria Completa de Benchmarks Multi-Etapas**:
> 1. **Fase 1 (Arquiteturas Mobile)**: ResNet18 Baseline, Two-Stage YOLO Crop, ResNet18 Coordinate Attention, MobileNetV4-Conv-Small e RepViT-M0.9.
> 2. **Fase 2 (Funções de Perda)**: Cross-Entropy com Label Smoothing, Focal Loss (gamma=2.0) e Balanced Softmax / Logit Adjustment (tau=1.0).
> 3. **Fase 3 & 4 (Paradigma P3 - SAM 2 Tap-to-Crop)**: Classificador treinado sobre a Região de Interesse (ROI) recortada de alta precisão pelo Segment Anything 2.
> 4. **Fase 5 (Avaliação em Vídeo & Comitê Temporal)**: Simulação do uso em campo no app Flutter (janela de 2s @ 3 FPS, regra de consenso >= 70%).
> 5. **Fase 6 (Otimização HPO)**: Optuna TPE com poda assíncrona ASHA (Successive Halving).
> 6. **Fase 7 (Deploy Mobile & Quantização)**: Exportação ONNX, TFLite Float16 e TFLite INT8 PTQ.
> 
> ⚡ **Vantagens da Execução Local**:
> - Uso direto do hardware local (RTX 2060 SUPER 8GB VRAM + 20 vCPUs + 32GB RAM);
> - Leitura instantânea no SSD NVMe sem latência de FUSE/Google Drive;
> - Zero risco de timeout de sessão ou consumo de Compute Units (CUs) do Colab.


### 1. Configuração Automática do Diretório Raiz do Repositório
Garante que o kernel do Jupyter opere a partir da raiz de `bio-interaction-ml`, independentemente da pasta onde o editor ou terminal foi aberto.

In [1]:
import os
from pathlib import Path

# Localiza automaticamente a raiz do projeto 'bio-interaction-ml'
cand = Path(os.getcwd()).resolve()
while cand != cand.parent:
    if (cand / "pyproject.toml").exists() and (cand / "base_tool").exists():
        break
    cand = cand.parent
else:
    cand = Path("/home/xsacbj/www/IFMaker/BioEcoInt/bio-interaction-ml").resolve()

# O comando mágico %cd ajusta tanto o Python quanto o shell '!' do Jupyter
%cd {cand}
print(f"✓ Diretório de trabalho ativo: {os.getcwd()}")


/home/xsacbj/www/IFMaker/BioEcoInt/bio-interaction-ml
✓ Diretório de trabalho ativo: /home/xsacbj/www/IFMaker/BioEcoInt/bio-interaction-ml


### 2. Diagnóstico do Hardware Local (GPU RTX 2060 SUPER)
Verifica a disponibilidade da GPU dedicada e memória de vídeo (VRAM) disponível.

In [2]:
import torch
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo: {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    allocated_vram = torch.cuda.memory_allocated(0) / (1024**2)
    print(f"VRAM Total: {total_vram:.2f} GB | VRAM Alocada: {allocated_vram:.2f} MB")
!nvidia-smi


CUDA disponível: True
Dispositivo: NVIDIA GeForce RTX 2060 SUPER
VRAM Total: 7.60 GB | VRAM Alocada: 0.00 MB
Thu Nov 12 17:46:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.99.02              Driver Version: 595.99.02      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2060 ...    Off |   00000000:01:00.0  On |                  N/A |
| 40%   36C    P8             19W /  175W |      36MiB /   8192MiB |     21%      Default |
|                              

### 3. Sincronização do Ambiente Virtual Local (`uv sync`)
Garante que todas as bibliotecas (`torch`, `torchvision`, `ultralytics`, `timm`, `optuna`) estão sincronizadas com o `pyproject.toml`.

In [3]:
!uv sync
!uv run python -c "import torch, torchvision, ultralytics, timm; print('✓ Ambiente local verificado! PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())"


Resolved 215 packages in 0.79ms
Uninstalled 20 packages in 67ms
 - asttokens==3.0.2
 - comm==0.2.3
 - debugpy==1.8.22
 - executing==2.3.0
 - ipykernel==7.4.0
 - ipython==9.18.1
 - ipython-pygments-lexers==1.1.1
 - jedi==0.20.1
 - jupyter-client==8.10.0
 - jupyter-core==5.9.1
 - matplotlib-inline==0.2.2
 - nest-asyncio2==1.7.4
 - parso==0.8.7
 - pexpect==4.9.0
 - ptyprocess==0.7.0
 - pure-eval==0.2.4
 - pyzmq==27.3.0
 - stack-data==0.6.3
 - tornado==6.6.1
 - traitlets==5.16.1
✓ Ambiente local verificado! PyTorch: 2.11.0+cu128 | CUDA: True


### 4. Verificação dos Datasets Locais (SSD NVMe)
Confirma que os dois datasets principais estão prontos para treinamento no diretório local `datasets/generated/`.

In [4]:
import os
from pathlib import Path

datasets_root = Path("datasets/generated")
sanitized_dir = datasets_root / "animals_video_sanitized"
sam2_dir = datasets_root / "animals_video_sam2_crops"

print(f"Dataset 1 (Sanitizado Full/Center-Crop): {'✓ Presente' if sanitized_dir.exists() else '❌ Ausente'}")
print(f"Dataset 2 (SAM 2 Tap-to-Crop):          {'✓ Presente' if sam2_dir.exists() else '❌ Ausente'}")

# Descompactação automática caso apenas o zip esteja presente no disco
if not sanitized_dir.exists() and (datasets_root / "animals_video_sanitized.zip").exists():
    print("Descompactando animals_video_sanitized.zip...")
    !unzip -q -o datasets/generated/animals_video_sanitized.zip -d datasets/generated/

if not sam2_dir.exists() and (datasets_root / "animals_video_sam2_crops.zip").exists():
    print("Descompactando animals_video_sam2_crops.zip...")
    !unzip -q -o datasets/generated/animals_video_sam2_crops.zip -d datasets/generated/

!ls -lh datasets/generated/


Dataset 1 (Sanitizado Full/Center-Crop): ✓ Presente
Dataset 2 (SAM 2 Tap-to-Crop):          ✓ Presente
total 944M
drwxr-xr-x 5 xsacbj sacbj 4,0K Nov 12 17:30 animals_video_sam2_crops
-rw-r--r-- 1 xsacbj sacbj 403M Nov 12 17:30 animals_video_sam2_crops.zip
-rw-r--r-- 1 xsacbj sacbj  113 Nov 12 17:30 animals_video_sam2_crops.zip.dvc
drwxr-xr-x 5 xsacbj sacbj 4,0K Nov 12 15:56 animals_video_sanitized
-rw-r--r-- 1 xsacbj sacbj 541M Nov 12 17:12 animals_video_sanitized.zip
-rw-r--r-- 1 xsacbj sacbj  112 Nov 12 17:12 animals_video_sanitized.zip.dvc


### 5. Bateria de Smoke Tests & Verificação de Vazamento (Pré-Voo)
Executa uma validação rápida de forward/backward em todos os modelos e confirma **zero data leakage** nos manifestos.

In [5]:
!uv run python tests/smoke_test_all_experiments.py
!uv run python tests/test_leakage_detector.py


Iniciando bateria de Smoke Tests para os Experimentos

---> Smoke test para configuração: options/exp1_resnet18_baseline.yml
  Step de treino concluído com sucesso. Loss: 2.8991, Acc: 0.5000
  Inferência concluída com sucesso. Shape: torch.Size([2, 16])

---> Smoke test para configuração: options/exp2_resnet18_yolo_crop.yml
  Step de treino concluído com sucesso. Loss: 2.3342, Acc: 0.5000
  Inferência concluída com sucesso. Shape: torch.Size([2, 16])

---> Smoke test para configuração: options/exp3_resnet18_ca.yml
  Step de treino concluído com sucesso. Loss: 2.8395, Acc: 0.0000
  Inferência concluída com sucesso. Shape: torch.Size([2, 16])

---> Smoke test para configuração: options/exp_mobilenetv4_small.yml
[LossFactory] Balanced Softmax ativado com tau=1.0 e contagens por classe: [344, 61, 366, 283, 450, 111, 298, 556, 753, 124, 438, 240, 549, 550, 741, 614]
  Step de treino concluído com sucesso. Loss: 6.2875, Acc: 0.0000
  Inferência concluída com sucesso. Shape: torch.Size([2, 16

### 6. 🚀 Experimento 1: ResNet18 Baseline Sanado
- **Arquitetura**: ResNet18 pré-treinada com Two-Stage Transfer Learning.
- **Divisão**: Vídeos estritamente disjuntos por classe (Zero Shuffle Leakage).
- **Otimizador**: AdamW com taxas de aprendizado diferenciais e RandAugment nativo na GPU.

In [6]:
!MPLBACKEND=Agg uv run base-train -opt options/exp1_resnet18_baseline.yml


2026-11-12 17:47:05,616 [INFO] PyTorch: 2.11.0+cu128 | GPU: True
2026-11-12 17:47:05,654 [INFO] Dataset de treino [ImageFolderClassificationDataset] criado. Total: 6478
2026-11-12 17:47:05,658 [INFO] Dataset de validacao [ImageFolderClassificationDataset] criado.
2026-11-12 17:47:05,888 [INFO] Modelo [ClassificationModel] construido.
2026-11-12 17:47:05,888 [INFO] Visualizador [training_plots] adicionado.
2026-11-12 17:47:05,888 [INFO] Visualizador [per_class_f1] adicionado.
2026-11-12 17:47:05,888 [INFO] Visualizador [confidence_box] adicionado.
2026-11-12 17:47:05,888 [INFO] Visualizador [confusion_matrix] adicionado.
2026-11-12 17:47:05,888 [INFO] Metrica [accuracy] adicionada.
2026-11-12 17:47:05,888 [INFO] Metrica [macro_f1] adicionada.
2026-11-12 17:47:05,888 [INFO] Metrica [balanced_acc] adicionada.
2026-11-12 17:47:05,888 [INFO] Iniciando treinamento por 20 epocas...
2026-11-12 17:47:11,115 [INFO] [Epoca 0][Iter 50] Train Metrics: {'loss': 2.63923716545105, 'acc': 0.125, 'lr_0'

### 7. 🎯 Experimento 2 (SOTA 1): Two-Stage YOLO Crop Classifier
- **Arquitetura**: Detector YOLOv8n localiza a ROI da fauna (+15% padding) -> ResNet18 classifica exclusivamente o animal.
- **Fix Aplicado**: YOLO mantido estritamente fora do `_modules` do PyTorch para eliminar vazamento no Model EMA e descompasso de device.

In [7]:
!MPLBACKEND=Agg uv run base-train -opt options/exp2_resnet18_yolo_crop.yml


2026-11-12 17:53:03,346 [INFO] PyTorch: 2.11.0+cu128 | GPU: True
2026-11-12 17:53:03,385 [INFO] Dataset de treino [ImageFolderClassificationDataset] criado. Total: 6478
2026-11-12 17:53:03,389 [INFO] Dataset de validacao [ImageFolderClassificationDataset] criado.
2026-11-12 17:53:03,618 [INFO] Modelo [ClassificationModel] construido.
2026-11-12 17:53:03,618 [INFO] Visualizador [training_plots] adicionado.
2026-11-12 17:53:03,618 [INFO] Visualizador [per_class_f1] adicionado.
2026-11-12 17:53:03,618 [INFO] Visualizador [confidence_box] adicionado.
2026-11-12 17:53:03,618 [INFO] Visualizador [confusion_matrix] adicionado.
2026-11-12 17:53:03,618 [INFO] Metrica [accuracy] adicionada.
2026-11-12 17:53:03,618 [INFO] Metrica [macro_f1] adicionada.
2026-11-12 17:53:03,618 [INFO] Metrica [balanced_acc] adicionada.
2026-11-12 17:53:03,619 [INFO] Iniciando treinamento por 20 epocas...
2026-11-12 17:53:15,421 [INFO] [Epoca 0][Iter 50] Train Metrics: {'loss': 2.570948600769043, 'acc': 0.125, 'lr_0

### 8. 🧠 Experimento 3 (SOTA 2): ResNet18 com Coordinate Attention (CA)
- **Arquitetura**: ResNet18 integrando blocos de Atenção Coordenada 2D (Hou et al., CVPR) nos estágios Layer3 e Layer4.
- **Benefício**: Codifica relações espaciais de longo alcance sem o custo quadrático de Vision Transformers.

In [8]:
!MPLBACKEND=Agg uv run base-train -opt options/exp3_resnet18_ca.yml


2026-11-12 18:06:48,833 [INFO] PyTorch: 2.11.0+cu128 | GPU: True
2026-11-12 18:06:48,872 [INFO] Dataset de treino [ImageFolderClassificationDataset] criado. Total: 6478
2026-11-12 18:06:48,876 [INFO] Dataset de validacao [ImageFolderClassificationDataset] criado.
2026-11-12 18:06:49,105 [INFO] Modelo [ClassificationModel] construido.
2026-11-12 18:06:49,106 [INFO] Visualizador [training_plots] adicionado.
2026-11-12 18:06:49,106 [INFO] Visualizador [per_class_f1] adicionado.
2026-11-12 18:06:49,106 [INFO] Visualizador [confidence_box] adicionado.
2026-11-12 18:06:49,106 [INFO] Visualizador [confusion_matrix] adicionado.
2026-11-12 18:06:49,106 [INFO] Metrica [accuracy] adicionada.
2026-11-12 18:06:49,106 [INFO] Metrica [macro_f1] adicionada.
2026-11-12 18:06:49,106 [INFO] Metrica [balanced_acc] adicionada.
2026-11-12 18:06:49,106 [INFO] Iniciando treinamento por 20 epocas...
2026-11-12 18:06:53,975 [INFO] [Epoca 0][Iter 50] Train Metrics: {'loss': 2.703343391418457, 'acc': 0.1015625, '

### 9. ⚡ Experimento 4 (SOTA 3): Google MobileNetV4-Conv-Small com Balanced Softmax & EMA
- **Arquitetura**: `mobilenetv4_conv_small.e2400_r224_in1k` (3.8M parâmetros, Google 2024).
- **Função de Perda**: **Balanced Softmax / Logit Adjustment** com tau=1.0 para mitigar o desbalanceamento de espécies raras.
- **Estabilidade**: Model EMA (beta=0.999), Differential LR e Weight Decay Seletivo.

In [9]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_mobilenetv4_small.yml


2026-11-12 18:12:56,714 [INFO] PyTorch: 2.11.0+cu128 | GPU: True
2026-11-12 18:12:56,751 [INFO] Dataset de treino [ImageFolderClassificationDataset] criado. Total: 6478
2026-11-12 18:12:56,755 [INFO] Dataset de validacao [ImageFolderClassificationDataset] criado.
2026-11-12 18:12:56,776 [INFO] Loading pretrained weights from Hugging Face hub (timm/mobilenetv4_conv_small.e2400_r224_in1k)
2026-11-12 18:12:57,136 [INFO] HTTP Request: HEAD https://huggingface.co/timm/mobilenetv4_conv_small.e2400_r224_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
2026-11-12 18:12:57,138 [INFO] [timm/mobilenetv4_conv_small.e2400_r224_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-11-12 18:12:57,148 [INFO] Missing keys (classifier.weight, classifier.bias) discovered while loading pretrained weights. This is expected if model is being adapted.
[LossFactory] Balanced Softmax ativado com tau=1.0 e contagens por classe: [344, 6

### 10. 🚀 Experimento 5 (SOTA 4): RepViT-M0.9 com Balanced Softmax & EMA
- **Arquitetura**: `repvit_m0_9` (4.7M parâmetros) com reparametrização estrutural no deploy mobile.
- **Otimização**: Balanced Softmax (tau=1.0), Model EMA e Two-Stage Warmup.

In [10]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_repvit_m0_9.yml


2026-11-12 18:18:06,598 [INFO] PyTorch: 2.11.0+cu128 | GPU: True
2026-11-12 18:18:06,636 [INFO] Dataset de treino [ImageFolderClassificationDataset] criado. Total: 6478
2026-11-12 18:18:06,640 [INFO] Dataset de validacao [ImageFolderClassificationDataset] criado.
2026-11-12 18:18:06,668 [INFO] Loading pretrained weights from Hugging Face hub (timm/repvit_m0_9.dist_300e_in1k)
2026-11-12 18:18:07,082 [INFO] HTTP Request: HEAD https://huggingface.co/timm/repvit_m0_9.dist_300e_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
2026-11-12 18:18:07,083 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-11-12 18:18:07,091 [INFO] [timm/repvit_m0_9.dist_300e_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-11-12 18:18:07,130 [INFO] Missing keys (head.head.l.weight, head.head.l.bias, head.head_dist.l.weight, head.hea

### 11. 🔬 Experimento 6 (Fase 2): Benchmark de Funções de Perda (Iso-condicional na MobileNetV4)
Compara sob a mesma arquitetura MobileNetV4-Conv-Small:
1. **Cross-Entropy Pura** com Label Smoothing (0.1);
2. **Focal Loss** (gamma=2.0) para alvos camuflados e desafiadores;
3. *(O experimento 4 acima serviu como a condição com Balanced Softmax tau=1.0)*.

In [11]:
# 1. Treino com Cross-Entropy pura + Label Smoothing 0.1
!MPLBACKEND=Agg uv run base-train -opt options/exp_loss_ce.yml

# 2. Treino com Focal Loss (gamma=2.0) + Label Smoothing 0.1
!MPLBACKEND=Agg uv run base-train -opt options/exp_loss_focal.yml


2026-11-12 18:18:47,393 [INFO] PyTorch: 2.11.0+cu128 | GPU: True
2026-11-12 18:18:47,432 [INFO] Dataset de treino [ImageFolderClassificationDataset] criado. Total: 6478
2026-11-12 18:18:47,436 [INFO] Dataset de validacao [ImageFolderClassificationDataset] criado.
2026-11-12 18:18:47,457 [INFO] Loading pretrained weights from Hugging Face hub (timm/mobilenetv4_conv_small.e2400_r224_in1k)
2026-11-12 18:18:47,804 [INFO] HTTP Request: HEAD https://huggingface.co/timm/mobilenetv4_conv_small.e2400_r224_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
2026-11-12 18:18:47,805 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-11-12 18:18:47,805 [INFO] [timm/mobilenetv4_conv_small.e2400_r224_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-11-12 18:18:47,814 [INFO] Missing keys (classifier.weight, classifier.bias)

### 12. 🎯 Experimento 7 (Fase 4): Paradigma P3 — Treinamento sobre ROI Recortada via SAM 2
- **Conceito**: Treina a MobileNetV4 diretamente sobre os recortes da fauna isolados pelo SAM 2 (+15% margem dinâmica).
- **Vantagem**: Reduz ruídos de fundo e foca nas texturas e contornos tridimensionais das esculturas.

In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_mobilenetv4_sam2_crops.yml


### 13. 📹 Experimento 8 (Fase 5): Benchmark de Avaliação em Vídeo & Comitê Temporal (2s @ 3 FPS)
Simula a rotina em campo no app Flutter: amostra 6 frames em 2 segundos e calcula a acurácia de consenso temporal (>= 70% dos votos) contra a predição isolada de quadro único.

In [ ]:
# 1. Comitê Temporal no modelo MobileNetV4 Direto
!uv run base-test-committee \
  -opt options/exp_mobilenetv4_small.yml \
  -c experiments/Exp_MobileNetV4_Small_BalancedSoftmax/models/best_macro_f1.pth \
  --threshold 0.70 \
  --fps 3.0 \
  --duration 2.0 \
  --output experiments/Exp_MobileNetV4_Small_BalancedSoftmax/committee_direct_report.json

# 2. Comitê Temporal no modelo treinado sobre recortes SAM 2
!uv run base-test-committee \
  -opt options/exp_mobilenetv4_sam2_crops.yml \
  -c experiments/Exp_MobileNetV4_SAM2_Crops/models/best_macro_f1.pth \
  --use-sam2 \
  --threshold 0.70 \
  --fps 3.0 \
  --duration 2.0 \
  --output experiments/Exp_MobileNetV4_SAM2_Crops/committee_sam2_report.json


### 14. ⚙️ Experimento 9 (Fase 6): Otimização de Hiperparâmetros HPO (Optuna + ASHA Pruning)
Busca bayesiana TPE com poda agressiva por Sucessivas Reduções (ASHA) no modelo campeão.

In [ ]:
!uv run python experiments/hpo_optuna_baseline.py \
  -opt options/exp_mobilenetv4_small.yml \
  --n-trials 15 \
  --epochs-per-trial 5


### 15. 📱 Exportação Mobile e Quantização (TFLite FP16 & INT8 PTQ)
Exporta os modelos campeões para ONNX, TFLite Float16 e TFLite INT8 completo.

In [ ]:
!MPLBACKEND=Agg uv run base-export -opt options/exp1_resnet18_baseline.yml
!MPLBACKEND=Agg uv run base-export -opt options/exp_mobilenetv4_small.yml
!MPLBACKEND=Agg uv run base-export -opt options/exp_mobilenetv4_sam2_crops.yml
!MPLBACKEND=Agg uv run base-export -opt options/exp_repvit_m0_9.yml


### 16. 📊 Comparativo Consolidado de Métricas de Todos os Experimentos
Gera tabela pandas comparativa com Macro-F1, Acurácia balanceada e relatórios do comitê temporal.

In [ ]:
import os
import glob
import json
import pandas as pd
import matplotlib.pyplot as plt

exp_dirs = [
    "experiments/Exp1_ResNet18_Baseline_Sanitized",
    "experiments/Exp2_ResNet18_YOLO_Crop",
    "experiments/Exp3_ResNet18_CoordinateAttention",
    "experiments/Exp_MobileNetV4_Small_BalancedSoftmax",
    "experiments/Exp_RepViT_M0_9_BalancedSoftmax",
    "experiments/Exp_Loss_CE_MobileNetV4",
    "experiments/Exp_Loss_Focal_MobileNetV4",
    "experiments/Exp_MobileNetV4_SAM2_Crops",
]

results = []
for d in exp_dirs:
    metrics_file = os.path.join(d, "val_metrics.json")
    log_files = glob.glob(os.path.join(d, "*.log"))
    exp_name = os.path.basename(d)
    if os.path.exists(metrics_file):
        with open(metrics_file) as f:
            data = json.load(f)
            results.append({"Experimento": exp_name, **data})
    elif log_files:
        results.append({"Experimento": exp_name, "Status": "Concluído (Ver log)"})

if results:
    df = pd.DataFrame(results)
    display(df)
else:
    print("Nenhum arquivo de métricas encontrado ainda.")

# Exibir relatórios do Comitê Temporal se disponíveis
committee_paths = [
    "experiments/Exp_MobileNetV4_Small_BalancedSoftmax/committee_direct_report.json",
    "experiments/Exp_MobileNetV4_SAM2_Crops/committee_sam2_report.json",
    "experiments/committee_mobilenetv4_direct.json",
    "experiments/committee_mobilenetv4_sam2.json",
]
for rep_path in committee_paths:
    if os.path.exists(rep_path):
        with open(rep_path) as f:
            rep_data = json.load(f)
        print(f"\n=== Relatório do Comitê: {rep_path} ===")
        print("Single-Frame Acc:", rep_data.get("single_frame_metrics", {}).get("accuracy"))
        print("Committee Acc (decided):", rep_data.get("committee_metrics", {}).get("accuracy_on_decided"))
        print("Abstention Rate:", rep_data.get("committee_metrics", {}).get("abstention_rate"))


### 17. 🔔 Notificação de Conclusão na Área de Trabalho
Dispara uma notificação nativa no sistema operacional avisando que os treinamentos terminaram.

In [ ]:
import os
try:
    os.system('notify-send "BioEcoInt Lab" "Bateria de Treinamentos e Benchmarks Locais Concluída com Sucesso!" --icon=dialog-information')
    print("🔔 Notificação enviada para a área de trabalho!")
except Exception as e:
    print(f"Aviso ao emitir notificação: {e}")
